# PolicyNav Notebook 1: Cloud Ingestion & DB Setup
This notebook handles scraping central government scheme landing pages for PDFs, downloading them directly into your Google Drive, and initializes an SQLite database for storing their metadata.


## 1. Setup Environment


In [1]:
!pip install requests beautifulsoup4 tqdm urllib3 -q
!apt-get update
!apt-get install -y wget
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install selenium


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,810 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:

## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


## 3. Setup Directory and Database


In [ ]:
import os
import sqlite3

# Create a dedicated directory in Google Drive
base_dir = '/content/drive/MyDrive/PolicyNav/documents'
os.makedirs(base_dir, exist_ok=True)

db_path = '/content/drive/MyDrive/PolicyNav/policies_meta.db'

try:
    conn = sqlite3.connect(db_path, timeout=10)
    conn.execute('PRAGMA journal_mode=WAL;')
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS policies
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  title TEXT,
                  url TEXT,
                  local_path TEXT)''')
    conn.commit()
    conn.close()
    print(f"Data directory: {base_dir}")
    print(f"Database setup at: {db_path}")
except Exception as e:
    print(f"DB Error: {e}")



Data directory: /content/drive/MyDrive/PolicyNav/documents
Database setup at: /content/drive/MyDrive/PolicyNav/policies_meta.db


## 4. Crawl & Download Policy PDFs


In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib.parse
from urllib.parse import urljoin
from tqdm import tqdm
import time
import urllib3
import os
import sqlite3
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Disable SSL warnings for government sites
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive",
}

base_dir = '/content/drive/MyDrive/PolicyNav/documents'
db_path = '/content/drive/MyDrive/PolicyNav/policies_meta.db'
db_lock = threading.Lock() # Ensures safe database writing across multiple threads

# Robust Session to handle timeouts
def get_session():
    session = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(HEADERS)
    session.verify = False
    return session

http = get_session()

# ==========================================
# 1. THE LINKS (Preserved & Categorized)
# ==========================================
policy_pages = {
    # Financial Inclusion & Social Security
    "PMJDY": "https://pmjdy.gov.in/",
    "PMSYM": "https://maandhan.in/",
    "AtalPensionYojana": "https://financialservices.gov.in/beta/en/atal-pension-yojna",
    "SukanyaSamriddhiYojana": "https://www.nsiindia.gov.in/InternalPage.aspx?Id_Pk=89",
    "MudraYojana": "https://www.mudra.org.in/",
    "StandUpIndia": "https://www.standupmitra.in/",

    # Health & Sanitation
    "PMJAY": "https://pmjay.gov.in/",
    "SwachhBharatMission": "https://swachhbharatmission.ddws.gov.in/",
    "MissionIndradhanush": "https://www.mohfw.gov.in/",

    # Agriculture & Farming
    "PMFBY": "https://pmfby.gov.in/",
    "PMKISAN": "https://pmkisan.gov.in/",
    "SoilHealthCard": "https://soilhealth.dac.gov.in/",

    # Rural Development & Water
    "JalJeevanMission": "https://jaljeevanmission.gov.in/",
    "PMGSY": "https://pmgsy.nic.in/",
    "GOBARdhan": "https://gobardhan.sbm.gov.in/",

    # Housing, Urban & Transport
    "PMAY_Urban": "https://pmay-urban.gov.in/",
    "SmartCitiesMission": "https://smartcities.gov.in/",
    "PM_eBus_Sewa": "https://pm-ebus-sewa.mohua.gov.in/",

    # Education & Skill Development
    "SkillIndia": "https://www.msde.gov.in/",
    "PMKVY_Official": "https://www.pmkvyofficial.org/",
    "PMPOSHAN": "https://pmposhan.education.gov.in/",

    # Technology & Digital
    "DigitalIndia": "https://digitalindia.gov.in/"
}

direct_pdfs = [
    ("PMJDY", "https://www.pmjdy.gov.in/files/E-Documents/PMJDY_Brochure_ENG.pdf"),
    ("Atal_Pension_Yojana","https://financialservices.gov.in/sites/default/files/APY_Guidelines.pdf"),
    ("Pradhan_Mantri_Mudra_Yojana","https://www.mudra.org.in/Content/Mudra_Scheme_Guidelines.pdf"),
    ("Stand_Up_India","https://www.standupmitra.in/Upload/Application/Documents/Stand-Up-India-Scheme-Guidelines.pdf"),
    ("PM_Shram_Yogi_Maandhan","https://maandhan.in/shramyogi/assets/images/PMSYM-Scheme-Details.pdf"),
    ("Sukanya_Samriddhi_Yojana","https://www.nsiindia.gov.in/InternalPage.aspx?Id_Pk=89"),
    ("SwachhBharatGramin", "https://archive.ids.ac.uk/clts/sites/communityledtotalsanitation.org/files/SwachhBharatMissionGraminGuidelines.pdf"),
    ("Mission_Indradhanush","https://www.mohfw.gov.in/pdf/MissionIndradhanushGuidelines.pdf"),
    ("National_Digital_Health_Mission","https://nha.gov.in/img/ndhm/National-Digital-Health-Blueprint.pdf"),
    ("PMFBY", "https://pmfby.gov.in/pdf/Revamped%20Operational%20Guidelines_17th%20August%202020.pdf"),
    ("PMKISAN", "https://pmkisan.gov.in/Documents/RevisedPM-KISANOperationalGuidelines(English).pdf"),
    ("SHC_Scheme", "https://www.agriwelfare.gov.in/sites/default/files/Guidelines_for_Demonstrations_under_SHC_Scheme.pdf"),
    ("JalJeevanMission", "https://jaljeevanmission.gov.in/sites/default/files/guideline/JJM_Operational_Guidelines.pdf"),
    ("PMGSY_Rural_Roads","https://pmgsy.nic.in/downloads/PMGSY_Guidelines.pdf"),
    ("GOBARdhan_Scheme","https://gobardhan.sbm.gov.in/assets/frontend/pdf/GOBARdhan-Guidelines.pdf"),
    ("National_Water_Policy","https://jalshakti-dowr.gov.in/sites/default/files/NationalWaterPolicy2012.pdf"),
    ("PMAY_Urban", "https://pmay-urban.gov.in/uploads/guidelines/Operational-Guidelines-of-PMAY-U.pdf"),
    ("PMAY_Gramin", "https://pmayg.dord.gov.in/netiayHome/Uploaded/Guidelines-English_Book_Final.pdf"),
    ("SmartCities", "https://mohua.gov.in/dataSmartCities/uploads/resource/resourceDoc/Resource_Doc_1723187622_Making_a_City_Smart_Learnings_from_the_Smart_Cities_Mission.pdf"),
    ("Electric_Bus_PMeBus_Sewa","https://pm-ebus-sewa.mohua.gov.in/assets/docs/PM-eBus-Sewa-Guidelines.pdf"),
    ("NEP", "https://www.education.gov.in/sites/upload_files/mhrd/files/NEP_Final_English_0.pdf"),
    ("PMKVY", "https://www.msde.gov.in/static/uploads/2024/02/PMKVY-4.0-Guidelines_final-copy.pdf"),
    ("PM_POSHAN_Mid_Day_Meal","https://pmposhan.education.gov.in/Files/Guidelines/PM_POSHAN_Guidelines.pdf"),
    ("Skill_India_Strategy","https://www.msde.gov.in/assets/images/Skill%20India%20Strategy%20Document.pdf"),
    ("National_Education_Policy_Implementation","https://www.education.gov.in/sites/upload_files/mhrd/files/NEP_Implementation_Strategy.pdf"),
    ("DigitalIndia", "https://www.meity.gov.in/static/uploads/2024/03/Brochure.pdf"),
    ("Startup_India_Action_Plan","https://dpiit.gov.in/sites/default/files/Startup_India_Action_Plan.pdf")
]

DRIVE_REGEX = r"(?:drive|docs)\.google\.com/(?:file/d/|open\?id=|uc\?id=|uc\?export=download&id=|viewer\?url=)([-_a-zA-Z0-9]+)"

# ==========================================
# 2. SELENIUM & LEVEL 2 SPIDERING
# ==========================================
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless=new')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument(f'user-agent={HEADERS["User-Agent"]}')
    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(15)
    return driver

def get_valuable_internal_links(source, base_url):
    """Level 2 Spider: Finds sub-pages likely to contain documents."""
    soup = BeautifulSoup(source, "html.parser")
    domain = urllib.parse.urlparse(base_url).netloc
    links = set()
    # Look for links that sound like document repositories
    keywords = ['doc', 'guide', 'polic', 'scheme', 'circular', 'download', 'resource']

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith(('javascript:', 'mailto:', 'tel:', '#')): continue
        full_url = urljoin(base_url, href)

        # Only stay on the exact same government domain
        if urllib.parse.urlparse(full_url).netloc == domain and full_url != base_url:
            if any(kw in href.lower() for kw in keywords) or full_url.endswith('.pdf'):
                links.add(full_url)

    # Limit to top 5 relevant sub-pages to prevent infinite crawling
    return list(links)[:5]

def extract_links_from_source(source, base_url):
    soup = BeautifulSoup(source, "html.parser")
    found_links = set()

    for link in soup.find_all("a", href=True):
        href = link["href"]
        if ".pdf" in href.lower():
            found_links.add(urljoin(base_url, href))

        drive_match = re.search(DRIVE_REGEX, href)
        if drive_match:
            found_links.add(f"https://drive.google.com/uc?export=download&id={drive_match.group(1)}")

    for iframe in soup.find_all("iframe", src=True):
        src = iframe.get("src", "")
        drive_match = re.search(DRIVE_REGEX, src)
        if drive_match:
            found_links.add(f"https://drive.google.com/uc?export=download&id={drive_match.group(1)}")

    return found_links

def find_pdfs_selenium(driver, page_url, depth=1):
    print(f"\nScanning: {page_url}")
    pdf_links = set()

    try:
        driver.get(page_url)
        time.sleep(4) # Wait for page JS to render

        # Grab Level 1 PDFs
        pdf_links.update(extract_links_from_source(driver.page_source, page_url))

        # Hunt for hidden "Download PDF" JavaScript buttons
        original_window = driver.current_window_handle
        download_buttons = driver.find_elements(By.XPATH, "//a[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'download pdf')]")

        for btn in download_buttons:
            try:
                href = btn.get_attribute("href")
                if not href or "javascript:" in str(href).lower() or "#" in str(href):
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(3)

                    if len(driver.window_handles) > 1:
                        for window_handle in driver.window_handles:
                            if window_handle != original_window:
                                driver.switch_to.window(window_handle)
                                pdf_links.update(extract_links_from_source(driver.page_source, driver.current_url))
                                driver.close()
                        driver.switch_to.window(original_window)
            except Exception:
                continue

        # --- LEVEL 2 SPIDERING TRIGGER ---
        if depth > 0:
            sub_pages = get_valuable_internal_links(driver.page_source, page_url)
            if sub_pages:
                print(f"  ↳ Deep Crawl: Found {len(sub_pages)} relevant sub-pages. Investigating...")
            for sub_link in sub_pages:
                # Call itself again with depth=0 to avoid infinite loops
                pdf_links.update(find_pdfs_selenium(driver, sub_link, depth=0))

    except Exception as e:
        print(f"Selenium failed or timed out for {page_url}: {e}")

    return list(pdf_links)

# ==========================================
# 3. SMART UPDATES & MULTI-THREADING
# ==========================================
def record_download(save_name, url, filepath):
    with db_lock: # Thread-safe database writing
        try:
            conn = sqlite3.connect(db_path, timeout=10)
            c = conn.cursor()
            c.execute("INSERT OR REPLACE INTO policies (title, url, local_path) VALUES (?, ?, ?)", (save_name, url, filepath))
            conn.commit()
            conn.close()
        except Exception:
            pass

def download_pdf(url, scheme_name):
    filename = url.split("/")[-1].split("?")[0]
    if not filename.endswith('.pdf') and "drive.google.com" not in url:
        filename += ".pdf"

    save_name = f"{scheme_name}_{filename}" if not filename.startswith(scheme_name) else filename
    filepath = os.path.join(base_dir, save_name)

    # --- UPDATE DETECTION (Delta Scraping) ---
    try:
        if os.path.exists(filepath):
            local_size = os.path.getsize(filepath)

            # Ping the server just to check the file size (does not download the whole file)
            head_req = http.head(url, allow_redirects=True, timeout=10)
            remote_size = int(head_req.headers.get("Content-Length", 0))

            # If the remote file is larger or smaller by more than 1KB, it's an update!
            if remote_size > 0 and abs(remote_size - local_size) > 1024:
                # Do nothing here, let the code below overwrite the file
                pass
            else:
                return # Skip, we already have the latest version
    except Exception:
        pass # If HEAD fails, proceed to attempt download anyway

    try:
        r = http.get(url, stream=True, timeout=15)
        if "application/pdf" not in r.headers.get("Content-Type", "").lower() and not r.content.startswith(b'%PDF'):
             return

        with open(filepath, "wb") as f:
             for chunk in r.iter_content(1024):
                 f.write(chunk)

        record_download(save_name, url, filepath)
        return True # Success marker for the progress bar
    except Exception as e:
        return False


# ==========================================
# EXECUTION
# ==========================================
print("Phase 1: Gathering all PDF links (including Deep Crawling)...")
pdf_mapping = []

for scheme_name, url in direct_pdfs:
    pdf_mapping.append((url, scheme_name))

driver = setup_driver()

for name, url in policy_pages.items():
    pdfs = find_pdfs_selenium(driver, url, depth=1)
    print(f"✅ Found {len(pdfs)} total PDFs for {name}")
    for pdf in pdfs:
        if pdf not in [u for u, s in pdf_mapping]:
            pdf_mapping.append((pdf, name))

driver.quit()

print(f"\nTotal PDFs to evaluate (Explicit + Scraped): {len(pdf_mapping)}")
print("Phase 2: Multi-Threaded Smart Downloading (Skipping existing, updating changed)...")

# --- MULTI-THREADED LAUNCHER ---
success_count = 0
with ThreadPoolExecutor(max_workers=5) as executor:
    # Submit all download tasks to the thread pool
    future_to_pdf = {executor.submit(download_pdf, url, scheme_name): url for url, scheme_name in pdf_mapping}

    # Show a master progress bar as threads complete their tasks
    for future in tqdm(as_completed(future_to_pdf), total=len(pdf_mapping), desc="Processing PDFs"):
        if future.result():
            success_count += 1

try:
    conn = sqlite3.connect(db_path, timeout=10)
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM policies")
    count = c.fetchone()[0]
    conn.close()
    print(f"\n✅ Done! {success_count} new/updated files downloaded.")
    print(f"🗄️ Total unique schemes registered in Database: {count}")
except Exception as e:
    print(f"Final DB Check Error: {e}")


Scanning: https://pmjdy.gov.in/
Found 7 PDFs for PMJDY

Scanning: https://pmjay.gov.in/
Found 0 PDFs for PMJAY

Scanning: https://pmay-urban.gov.in/
Found 37 PDFs for PMAY_Urban

Scanning: https://pmfby.gov.in/
Found 0 PDFs for PMFBY

Scanning: https://pmkisan.gov.in/
Failed: HTTPSConnectionPool(host='pmkisan.gov.in', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x787266726ab0>, 'Connection to pmkisan.gov.in timed out. (connect timeout=25)'))
Found 0 PDFs for PMKISAN

Scanning: https://jaljeevanmission.gov.in/
Found 35 PDFs for JalJeevanMission

Scanning: https://digitalindia.gov.in/
Failed: 403 Client Error: Forbidden for url: https://digitalindia.gov.in/
Found 0 PDFs for DigitalIndia

Scanning: https://www.msde.gov.in/
Found 0 PDFs for SkillIndia

Scanning: https://maandhan.in/
Failed: HTTPSConnectionPool(host='maandhan.in', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.

SwachhBharatMission_SSG_Framework.pdf:  21%|██        | 13.9M/66.1M [04:08<15:41, 55.5kB/s]

In [21]:
import time
import urllib.parse
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless=new')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--lang=en-US')
    prefs = {"intl.accept_languages": "en-US,en"}
    chrome_options.add_experimental_option("prefs", prefs)
    chrome_options.add_argument(f'user-agent={HEADERS["User-Agent"]}')

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(30)
    return driver

def clear_obstructions_safely(driver):
    """Nukes Bhashini and clicks cookies without destroying the page."""
    try: driver.switch_to.alert.accept()
    except: pass
    try:
        driver.execute_script("""
            let buttons = document.querySelectorAll('button, a');
            for(let btn of buttons) {
                let txt = (btn.innerText || '').trim().toLowerCase();
                if(txt === 'accept all cookies' || txt === 'accept cookies' || txt === 'ok') {
                    if (btn.offsetWidth > 0) btn.click();
                }
            }
            let elements = document.querySelectorAll('[id*="bhashini"], [class*="bhashini"]');
            elements.forEach(el => { el.style.display = 'none'; });
        """)
    except: pass

def get_hybrid_internal_links(source, base_url):
    """Identifies document repositories by scoring visible text and URL paths."""
    soup = BeautifulSoup(source, "html.parser")
    domain = urllib.parse.urlparse(base_url).netloc.replace('www.', '')
    link_scores = {}
    magic_words = ['document', 'polic', 'scheme', 'circular', 'guideline', 'upload', 'act', 'rule', 'resource', 'download', 'report', 'publication', 'notice']

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith(('javascript:', 'mailto:', 'tel:', '#')): continue
        full_url = urljoin(base_url, href)
        if domain not in urllib.parse.urlparse(full_url).netloc or full_url == base_url: continue

        visible_text = a.get_text(separator=' ', strip=True).lower()
        url_lower = full_url.lower()

        score = 0
        for word in magic_words:
            if word in url_lower: score += 1
            if word in visible_text: score += 2
        if score > 0:
            link_scores[full_url] = max(link_scores.get(full_url, 0), score)

    return sorted(link_scores.keys(), key=lambda k: link_scores[k], reverse=True)[:10]

def extract_pdfs_from_html(source, base_url):
    soup = BeautifulSoup(source, "html.parser")
    found_links = set()
    for link in soup.find_all("a", href=True):
        href = link["href"].lower()
        full_url = urljoin(base_url, link["href"])
        if ".pdf" in href or "/upload" in href or "/file" in href:
            found_links.add(full_url)
    return found_links

def scan_table_with_pagination(driver, start_url):
    """The battle-tested pagination and nested folder logic."""
    try:
        driver.get(start_url)
        time.sleep(5)
        clear_obstructions_safely(driver)
        # Trigger lazy-loading
        driver.execute_script("window.scrollTo(0, 800);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(5)
    except: return set(), []

    all_pdfs = set()
    nested_folders = []
    page_num = 1

    while True:
        clear_obstructions_safely(driver)
        page_source = driver.page_source

        # 1. Extract PDFs
        current_batch = extract_pdfs_from_html(page_source, driver.current_url)
        all_pdfs.update(current_batch)

        # 2. Extract 'VIEW ALL' links (Nested Categories)
        soup = BeautifulSoup(page_source, "html.parser")
        for a in soup.find_all("a", href=True):
            txt = a.get_text(separator=' ', strip=True).lower()
            if "view all" in txt or "view more" in txt:
                folder_url = urljoin(driver.current_url, a["href"])
                if 'javascript' not in folder_url and '#' not in folder_url and folder_url not in nested_folders:
                    nested_folders.append(folder_url)

        # 3. Crash-Proof Pagination Icon-Hunter
        clicked_next = driver.execute_script("""
            let nextBtn = null;
            let candidates = document.querySelectorAll('a, button, li');
            for(let el of candidates) {
                let txt = (el.innerText || '').trim().toLowerCase();
                let aria = String(el.getAttribute('aria-label') || '').toLowerCase();
                let cls = String(el.getAttribute('class') || '').toLowerCase();
                if(cls.includes('disabled') || el.hasAttribute('disabled')) continue;
                if(txt === '>' || txt === '›' || txt === '»' || txt === 'next' || aria.includes('next') || cls.includes('next')) {
                    if(el.offsetWidth > 0 && el.offsetHeight > 0) {
                        nextBtn = el;
                        let childA = el.querySelector('a');
                        if(childA) nextBtn = childA;
                        break;
                    }
                }
            }
            if (!nextBtn) {
                let svgs = document.querySelectorAll('svg, i, span');
                for (let icon of svgs) {
                    let cls = String(icon.getAttribute('class') || '').toLowerCase();
                    if (cls.includes('right') || cls.includes('next') || cls.includes('chevron')) {
                        let parent = icon.closest('a, button, li');
                        if (parent && !String(parent.getAttribute('class') || '').toLowerCase().includes('disabled') && parent.offsetWidth > 0) {
                            nextBtn = parent;
                            let childA = parent.querySelector('a');
                            if(childA) nextBtn = childA;
                            break;
                        }
                    }
                }
            }
            if (nextBtn) { nextBtn.click(); return true; }
            return false;
        """)

        if clicked_next:
            page_num += 1
            time.sleep(4)
        else: break

    return all_pdfs, nested_folders

def master_pipeline(driver, homepage_url):
    print(f"🚀 Starting Master Pipeline for: {homepage_url}\n")
    final_collection = set()

    try:
        driver.get(homepage_url)
        time.sleep(5) # Give homepage ample time to render menus
        clear_obstructions_safely(driver)
        # Scroll to wake up JS menus
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(2)

        final_collection.update(extract_pdfs_from_html(driver.page_source, homepage_url))

        repo_links = get_hybrid_internal_links(driver.page_source, homepage_url)
        print(f"🔍 Identified {len(repo_links)} potential repositories.\n")

        visited = set()
        queue = list(repo_links)

        while queue:
            current_target = queue.pop(0)
            if current_target in visited: continue
            visited.add(current_target)

            print(f"    Scanning: {current_target}")
            pdfs, extras = scan_table_with_pagination(driver, current_target)
            final_collection.update(pdfs)
            for item in extras:
                if item not in visited: queue.append(item)

    except Exception as e:
        print(f"Error: {e}")

    return final_collection

# ==========================================
# EXECUTION
# ==========================================
target_site = "https://www.msde.gov.in/"
driver = setup_driver()
results = master_pipeline(driver, target_site)
driver.quit()

print(f"\n✅ Total unique documents found: {len(results)}")
if results:
    print("\nSample discovery:")
    for i, url in enumerate(list(results)[:10]):
        print(f"  {i+1}. {url}")

🚀 Starting Master Pipeline for: https://www.msde.gov.in/

🔍 Identified 10 potential repositories.

    Scanning: https://www.msde.gov.in/documents/act-and-policies
    Scanning: https://www.msde.gov.in/documents/orders-and-notices
    Scanning: https://www.msde.gov.in/documents/publications
    Scanning: https://www.msde.gov.in/documents/guidelines
    Scanning: https://www.msde.gov.in/documents
    Scanning: https://www.msde.gov.in/policies
    Scanning: https://www.msde.gov.in/offerings
    Scanning: https://www.msde.gov.in/connect
    Scanning: https://www.msde.gov.in/documents/press-release
    Scanning: https://www.msde.gov.in/documents/gazettes-notifications
    Scanning: https://xn--o1bna6ezc1cxc.xn--11b7cb3a6a.xn--h2brj9c/documents/act-and-policies/apprentices-IjM1ETMtQWa?pageTitle=The-Apprentices-Act,-1961

✅ Total unique documents found: 83

Sample discovery:
  1. https://www.msde.gov.in/static/uploads/2024/02/DoPT-Notification-regarding-Special-Leave-during-SH-Inquiry-1.pdf
